# 워크북 1 — 학생 실습용
노션의 실습 0~5에 제시된 코드를 참고해 각 TODO를 완성하고 차례대로 실행하세요.


## 실습 0. 환경 설정
아래 코드를 노션에서 복사하여 Colab의 [코드] 셀에 입력합니다. 실행 후 설치 오류가 없는지 확인합니다.
예상 결과: 설치 완료 (경고는 환경에 따라 다를 수 있음)



In [ ]:
%pip -q install pypdf scikit-learn


## 실습 1. 제공 PDF 업로드·텍스트 추출
Drive 7주차 ZIP 안의 data/week07_course_guide.pdf를 Colab 파일에 업로드합니다. TODO에는 PDF 파일 이름을 입력합니다.
예상 결과: 텍스트가 있는 페이지: 2 (제공 PDF 기준)



In [ ]:
from pathlib import Path
from pypdf import PdfReader

pdf_name = "TODO_파일명.pdf"  # TODO 1: 제공 PDF 파일명
pdf_path = Path(pdf_name)
assert pdf_path.exists(), "Colab에 PDF 파일을 먼저 업로드하세요."

reader = PdfReader(str(pdf_path))
records = []
for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()
    if text:
        records.append({"page": page_number, "text": text})

assert records, "추출 텍스트가 없습니다. 스캔 PDF라면 OCR이 필요합니다."
print("텍스트가 있는 페이지:", len(records))


## 실습 2. 청크 생성
문서를 일정 단어 수로 나눕니다. TODO 2에서 size=75, overlap=20을 완성합니다.
예상 결과: 생성 청크: PDF 내용·추출 공백에 따른 정수



In [ ]:
def chunk_words(text, size=75, overlap=20):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("0 <= overlap < size 조건을 확인하세요.")
    words = text.split()
    step = size - overlap
    return [
        " ".join(words[i:i + size])
        for i in range(0, len(words), step)
        if words[i:i + size]
    ]

chunk_size = None  # TODO 2: 75
overlap = None  # TODO 2: 20
chunks = [
    {"page": item["page"], "text": part}
    for item in records
    for part in chunk_words(item["text"], chunk_size, overlap)
]
print("생성 청크:", len(chunks))
print("첫 청크:", chunks[0]["text"][:100])


## 실습 3. TF-IDF 문서 벡터 생성
한글에도 적용하기 쉬운 문자 단위 2~4그램 TF-IDF를 사용합니다. TODO에서 analyzer와 ngram_range를 완성합니다.
예상 결과: (청크 수, TF-IDF 특징 수) 형태



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer="word"  # TODO 3: char로 변경,
    ngram_range=(1, 1)  # TODO 3: (2, 4)로 변경
)
document_vectors = vectorizer.fit_transform(
    [part["text"] for part in chunks]
)
print("행렬 크기:", document_vectors.shape)


## 실습 4. 질문 검색 함수
질문을 같은 벡터 공간에 변환하고 코사인 유사도 순으로 상위 청크를 반환합니다. TODO에서 similarity 함수를 채웁니다.
예상 결과: 상위 두 개의 페이지, 소수점 3자리 점수, 근거 텍스트



In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(question, top_k=3):
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(  # TODO 4: 유사도 계산 함수를 설명하세요

        question_vector, document_vectors
    ).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "page": chunks[int(i)]["page"],
            "text": chunks[int(i)]["text"],
            "score": float(scores[int(i)])
        }
        for i in order
    ]

results = retrieve("RAG가 무엇인가요?", top_k=2)
for item in results:
    print(f"페이지 {item['page']} / 점수 {item['score']:.3f}")
    print(item["text"][:100])


## 실습 5. 결과표·청크 크기 비교
3개 질문의 최고 검색 결과를 표로 만들고, size=45로 바꿔 실습 2~5를 재실행합니다.
예상 결과: 3행의 질문·페이지·유사도 테이블



In [ ]:
import pandas as pd

questions = [
    "RAG가 무엇인가요?",
    "TODO_두번째 질문을 입력하세요",
    "청크를 왜 만드나요?"
]
rows = []
for question in questions:
    hit = retrieve(question, top_k=1)[0]
    rows.append({
        "질문": question,
        "페이지": hit["page"],
        "유사도": round(hit["score"], 3)
    })
result_table = pd.DataFrame(rows)
display(result_table)


## 결과 제출
모든 셀을 실행한 후 결과표와 실행화면을 캡처하고 TODO를 완성한 노트북을 제출하세요.
